# Hardik LLM Lab - Google Colab T4 Setup

This notebook sets up the hardik-llm-lab environment on Google Colab with T4 GPU.

**Usage**:
1. Open this notebook in Colab
2. Runtime → Change runtime type → GPU → T4
3. Run all cells sequentially
4. Your local VS Code can connect via SSH tunnel (see cell 4)

In [1]:
# @title 1. System Check & GPU Verification
import torch
import subprocess
import sys

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN: {torch.backends.cudnn.version()}")
    # Memory info
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    # Quick tensor test
    x = torch.randn(1000, 1000, device='cuda')
    y = x @ x.T
    print(f"GPU compute test: OK (matmul {y.shape})")
else:
    print("⚠️  No GPU detected! Change runtime to GPU → T4")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA Version: 13.0
cuDNN: 92700
VRAM: 15.6 GB
GPU compute test: OK (matmul torch.Size([1000, 1000]))


In [ ]:
# @title 2. Clone & Install Repository
import os
import subprocess

REPO_URL = "https://github.com/Hardik-Sankhla/hardik-llm-lab.git"
REPO_DIR = "/content/hardik-llm-lab"

# Clone if not exists
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository exists, pulling latest...")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print(f"Working in: {os.getcwd()}")

# Install in development mode
print("Installing package in development mode...")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev,experiments]"], check=True)

# Verify import
import hardik_llm
print(f"✅ hardik_llm v{hardik_llm.__version__} imported successfully")
print(f"Available: {hardik_llm.__all__}")

Cloning https://github.com/Hardik-Sankhla/hardik-llm-lab.git...
Working in: /content/hardik-llm-lab
Installing package in development mode...


In [ ]:
# @title 3. Data Setup (Optional - Download Datasets)
import os
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

# Example: Download WikiText-2 for tokenizer experiments
# Uncomment to download
# import urllib.request
# urls = {
#     "wikitext-2-train": "https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-2-v1.zip",
# }
# for name, url in urls.items():
#     path = DATA_DIR / f"{name}.zip"
#     if not path.exists():
#         print(f"Downloading {name}...")
#         urllib.request.urlretrieve(url, path)
#     # Extract
#     import zipfile
#     with zipfile.ZipFile(path, 'r') as zf:
#         zf.extractall(DATA_DIR)

print(f"Data directory: {DATA_DIR}")
print(f"Contents: {list(DATA_DIR.iterdir())}")

In [ ]:
# @title 4. VS Code Remote SSH Tunnel Setup
# Run this cell to get the SSH command for VS Code Remote-SSH

import subprocess
import os

# Install cloudflared for tunnel (if not present)
def install_cloudflared():
    try:
        subprocess.run(["cloudflared", "--version"], capture_output=True, check=True)
        print("cloudflared already installed")
    except:
        print("Installing cloudflared...")
        subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=True)
        subprocess.run(["sudo", "dpkg", "-i", "cloudflared-linux-amd64.deb"], check=True)

install_cloudflared()

# Start SSH server
print("Starting SSH server on port 2222...")
subprocess.run(["sudo", "apt-get", "update"], capture_output=True)
subprocess.run(["sudo", "apt-get", "install", "-y", "openssh-server"], capture_output=True)
subprocess.run(["sudo", "mkdir", "-p", "/var/run/sshd"])
subprocess.run(["sudo", "sed", "-i", "'s/#Port 22/Port 2222/'", "/etc/ssh/sshd_config"])
subprocess.run(["sudo", "sed", "-i", "'s/#PermitRootLogin prohibit-password/PermitRootLogin yes/'", "/etc/ssh/sshd_config"])
subprocess.run(["sudo", "sed", "-i", "'s/#PasswordAuthentication yes/PasswordAuthentication yes/'", "/etc/ssh/sshd_config"])
# Set root password for SSH
import random
import string
pwd = ''.join(random.choices(string.ascii_letters + string.digits, k=16))
subprocess.run([f"echo 'root:{pwd}'"], shell=True, check=True)
subprocess.run(["sudo", "chpasswd"], input=f"root:{pwd}".encode(), check=True)
subprocess.run(["sudo", "service", "ssh", "start"])

# Start cloudflare tunnel
print("Starting Cloudflare tunnel...")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "ssh://localhost:2222"], 
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for tunnel URL
import time
time.sleep(5)
output = tunnel_proc.stderr.read().decode()
print(output)

# Extract tunnel URL
import re
match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', output)
if match:
    tunnel_url = match.group(1)
    print(f"\n🔗 Tunnel URL: {tunnel_url}")
    print(f"\n📋 VS Code SSH Config:")
    print(f"Host colab-t4")
    print(f"    HostName {tunnel_url.replace('https://', '')}")
    print(f"    User root")
    print(f"    Port 22")
    print(f"    ProxyCommand cloudflared access ssh --hostname %h")
    print(f"\n🔑 Root password: {pwd}")
    print(f"\n📝 Add above to ~/.ssh/config in VS Code, then Connect to 'colab-t4'")
else:
    print("Tunnel URL not found in output. Check cloudflared logs.")

In [ ]:
# @title 5. Quick Training Test (Verify Everything Works)
import torch
from hardik_llm.model import GPTConfig, GPTModel
from hardik_llm.tokenizer import BPETokenizer, TokenizerConfig

# Tiny model for quick test
config = GPTConfig(
    vocab_size=1000,
    d_model=128,
    n_layers=2,
    n_heads=4,
    max_seq_len=256,
    dropout=0.1
)
model = GPTModel(config).cuda()
print(f"Model params: {model.get_num_params():,}")

# Dummy forward pass
x = torch.randint(0, 1000, (2, 64), device='cuda')
logits, loss = model(x, x)
print(f"Forward pass OK: logits={logits.shape}, loss={loss.item():.4f}")

# Generation test
generated = model.generate(x[:1], max_new_tokens=10, temperature=1.0)
print(f"Generation OK: {generated.shape}")

print("\n✅ All systems operational on T4!")

In [ ]:
# @title 6. Experiment Runner Helper
def run_experiment(exp_id: str, config_path: str = None, **kwargs):
    """Run an experiment from the lab."""
    import subprocess
    import sys
    
    cmd = [sys.executable, "-m", "hardik_llm.experiments.run", "--exp-id", exp_id]
    if config_path:
        cmd.extend(["--config", config_path])
    for k, v in kwargs.items():
        cmd.extend([f"--{k}", str(v)])
    
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd="/content/hardik-llm-lab")
    return result.returncode == 0

# Example usage:
# run_experiment("EXP-001", "experiments/configs/EXP-001-tokenizer-vocab-size.yaml")

In [ ]:
# @title 7. Sync Changes Back to GitHub (Run Before Disconnecting)
def sync_to_github(commit_message: str = "Colab: update from T4 session"):
    """Commit and push changes from Colab to GitHub."""
    import subprocess
    import os
    
    os.chdir("/content/hardik-llm-lab")
    
    # Check for changes
    result = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if not result.stdout.strip():
        print("No changes to commit")
        return
    
    print("Changes detected:")
    print(result.stdout)
    
    # Add all
    subprocess.run(["git", "add", "-A"])
    
    # Commit
    subprocess.run(["git", "commit", "-m", commit_message])
    
    # Push (requires PAT or SSH key configured)
    # For HTTPS with token: git remote set-url origin https://<TOKEN>@github.com/Hardik-Sankhla/hardik-llm-lab.git
    push_result = subprocess.run(["git", "push", "origin", "main"])
    
    if push_result.returncode == 0:
        print("✅ Pushed to GitHub successfully")
    else:
        print("⚠️ Push failed. Configure git credentials:")
        print("   git remote set-url origin https://<YOUR_PAT>@github.com/Hardik-Sankhla/hardik-llm-lab.git")
        print("   Or use SSH: git remote set-url origin git@github.com:Hardik-Sankhla/hardik-llm-lab.git")

# Usage:
# sync_to_github("feat: implemented BPE tokenizer + EXP-001 results")

In [ ]:
# @title 8. Keep-Alive & Monitoring (Run in Background)
import threading
import time
import torch

def gpu_monitor(interval=30):
    """Print GPU memory usage periodically."""
    while True:
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1e9
            reserved = torch.cuda.memory_reserved() / 1e9
            print(f"[GPU] Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB")
        time.sleep(interval)

# Start monitor in background thread
monitor_thread = threading.Thread(target=gpu_monitor, args=(60,), daemon=True)
monitor_thread.start()
print("GPU monitor started (logs every 60s)")

# Prevent Colab disconnect (run in separate cell if needed)
# from IPython.display import Javascript
# display(Javascript(''')
# function ClickConnect(){
#   console.log("Keep alive");
#   document.querySelector("colab-connect-button").click()
# }
# setInterval(ClickConnect, 30000)
# '''))